# Task 02 — Silver: Orders Clean & Enrich

**Workshop: Final Pipeline | Layer 2 of 3**

## Goal

Read `bronze.lab_orders`, apply five transformations, write to `silver.lab_orders`.

```
bronze.lab_orders
        |
        v  cast * withColumn * when/otherwise * filter * drop
        v
silver.lab_orders
```

## Transformations to implement

| # | What | Details |
|---|------|---------|
| 1 | `cast` | `order_datetime` String -> TimestampType |
| 2 | `withColumn` | Add `net_amount` = `total_amount` x (1 - `discount_percent` / 100), rounded to 2 dp |
| 3 | `when / otherwise` | Add `order_tier`: `"Premium"` >= 500 * `"Standard"` >= 200 * `"Small"` otherwise |
| 4 | `filter` | Remove rows where `order_id` IS NULL |
| 5 | `drop` | Remove `_source_file` (Bronze metadata, not needed in Silver) |

> `net_amount` is used by Task 03 for revenue — make sure the formula is correct.


In [ ]:
%run ../../setup/00_setup

In [ ]:
dbutils.widgets.text("catalog",       CATALOG,       "Catalog")
dbutils.widgets.text("bronze_schema", BRONZE_SCHEMA, "Bronze Schema")
dbutils.widgets.text("silver_schema", SILVER_SCHEMA, "Silver Schema")

catalog       = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

source_table = f"{catalog}.{bronze_schema}.lab_orders"
target_table = f"{catalog}.{silver_schema}.lab_orders"

print(f"Source : {source_table}")
print(f"Target : {target_table}")

---

## Exercise 1 — Imports

Import all PySpark functions needed for the five transformations above.


**💡 Guidance — Exercise 1**

Import the transformation functions needed for the five Silver operations.

**Key functions to import**
- `col` — reference a DataFrame column by name in expressions
- `to_timestamp(col_expr, format)` — parse a string column into `TimestampType` using a Java format pattern. For this dataset use `"yyyy-MM-dd'T'HH:mm:ss"`. The single quotes around `T` escape the literal character in the pattern.
- `when(cond, val).when(...).otherwise(val)` — conditional expression (SQL `CASE WHEN`); always end the chain with `.otherwise()` to cover all rows
- `round(col_expr, scale)` — rounds a `Column` expression to N decimal places. The first argument **must** be a `Column`, not a Python float.
- `lit(value)` — wraps a Python scalar as a constant `Column` expression. Required when mixing Python numbers with column arithmetic: `col("a") * (lit(1) - col("b") / lit(100))`

**Important note on shadowing**
Importing `round` and `sum` from PySpark shadows Python's built-ins of the same names. This is intentional — the PySpark versions operate on `Column` expressions, not Python scalars.

**Things to think about**
- What is the difference between `round(2.345, 2)` (Python built-in) and `round(col("amount"), 2)` (PySpark `Column`)?
- Why do we need `lit(1)` instead of just `1` when writing `col("total") * (lit(1) - col("discount") / lit(100))`?

In [ ]:
# YOUR CODE HERE

---

## Exercise 2 — Read the Bronze table


**💡 Guidance — Exercise 2**

Read the Bronze `lab_orders` table into a DataFrame before applying transformations.

**Reading with spark.table()**
`spark.table("catalog.schema.table_name")` is the cleanest way to read a managed Delta table from Unity Catalog. It is equivalent to `spark.sql("SELECT * FROM ...")` but more concise in a Python chain. No file path is needed — Unity Catalog manages the location.

The Bronze table was written by Task 01 Notebook — that notebook must have run successfully before you can execute this cell. Store the result as `orders_bronze`.

**Expected state of the Bronze data**
At this point `orders_bronze` should contain the 11 original JSON columns plus `_load_ts` and `_source_file`. The `order_datetime` column is still a raw string — you will cast it in Exercise 3.

**Things to think about**
- When would you use `spark.read.format("delta").load("/path")` instead of `spark.table("name")`?
- What happens to the Silver transformation chain if the Bronze table is empty?

In [ ]:
# Read the Bronze table into orders_bronze (Task 01 must have run first)
orders_bronze = # YOUR CODE HERE

In [ ]:
print(f"Bronze rows : {orders_bronze.count():,}")
orders_bronze.printSchema()

---

## Exercise 3 — Apply all five transformations in one chain

Chain all transformations into a single expression. Name the result `orders_silver`.


**💡 Guidance — Exercise 3**

Apply all five Silver transformations in a single chained expression to produce `orders_silver`.

**Transformation 1 — Cast the timestamp**
`order_datetime` is currently a raw string. Overwrite it using `.withColumn("order_datetime", to_timestamp(col("order_datetime"), "yyyy-MM-dd'T'HH:mm:ss"))`. The `'T'` in the format pattern is a quoted literal character.

**Transformation 2 — Net amount**
Net amount = `total_amount × (1 − discount_percent / 100)`. Use column arithmetic with `lit()` for the Python scalars: `round(col("total_amount") * (lit(1) - col("discount_percent") / lit(100)), 2)`. Name the column `net_amount`.

**Transformation 3 — Order tier**
Use `when().when().otherwise()` on `net_amount` (not `total_amount`): `≥ 500 → "Premium"`, `≥ 200 → "Standard"`, otherwise `"Small"`. Place the most restrictive condition first.

**Transformation 4 — Filter nulls**
Remove rows where `order_id` is null using `.filter(col("order_id").isNotNull())`.

**Transformation 5 — Drop metadata column**
Remove `_source_file` with `.drop("_source_file")`.

**Things to think about**
- Does the order of these transformations matter? What would break if the null filter came before the timestamp cast?
- Why is `round(avg(...), 2)` nested — what type does `avg(...)` return?

In [ ]:
# Apply all 5 transformations in a single chain:
#  1. cast order_datetime: String → TimestampType (format: "yyyy-MM-dd'T'HH:mm:ss")
#  2. add net_amount = round(total_amount * (1 - discount_percent / 100), 2)
#  3. add order_tier: "Premium" >= 500 | "Standard" >= 200 | "Small" otherwise
#  4. filter rows where order_id IS NOT NULL
#  5. drop _source_file
orders_silver = (
    orders_bronze
    # YOUR CODE HERE
)

In [ ]:
print(f"Silver rows : {orders_silver.count():,}")
orders_silver.select(
    "order_id","order_datetime","total_amount",
    "discount_percent","net_amount","order_tier"
).show(10, truncate=False)
orders_silver.printSchema()

---

## Exercise 4 — Write to Silver as a managed Delta table


**💡 Guidance — Exercise 4**

Write `orders_silver` to a managed Delta table as the Silver layer of the pipeline.

**Write options**
Use `mode("overwrite")` — Silver is always fully rebuilt from Bronze on each pipeline run. Set `option("overwriteSchema", "true")` because the Silver schema contains new columns (`net_amount`, `order_tier`) that were not present in Bronze. Without this option, Spark raises an `AnalysisException` if the target table already exists with a different schema from a previous run.

**Full refresh pattern**
Rebuilding Silver entirely on each run ensures that any fix applied to the Bronze data propagates through automatically — no incremental state to manage.

**Things to think about**
- Why is Silver always rebuilt from Bronze rather than updated incrementally?
- What would happen if you saved `orders_silver` before applying the null filter in Exercise 3?

In [ ]:
# Write orders_silver to target_table as a managed Delta table
# mode: overwrite | overwriteSchema: true (Silver adds new columns not in Bronze)
# YOUR CODE HERE

In [ ]:
row_count = spark.table(target_table).count()
print(f"OK  {target_table}  ->  {row_count:,} rows")
display(spark.table(target_table).limit(5))

In [ ]:
import json
dbutils.notebook.exit(json.dumps({"status":"SUCCESS","table":target_table,"rows":row_count}))